In [31]:
!pip install pandas
!pip install PyMuPDF

You should consider upgrading via the '/Users/rachelwsakamoto/jupyter-env/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/rachelwsakamoto/jupyter-env/bin/python3 -m pip install --upgrade pip' command.


In [32]:
import pandas as pd

df = pd.read_csv('data/ucsd_all_grades.csv', encoding='utf-8-sig')

print(df.head())

df.columns = df.columns.str.strip()

  Subject Course  Year Quarter                  Title  \
0    ELWR      1    16      FA  Entry Level Writing 1   
1    ELWR      1    16      FA  Entry Level Writing 1   
2    ELWR      1    16      FA  Entry Level Writing 1   
3    ELWR      1    16      FA  Entry Level Writing 1   
4    ELWR      1    16      FA  Entry Level Writing 1   

                Instructor    GPA      A      B      C    D   F      W   P  NP  
0    Ajo, Frances Margaret  2.659  12.8%  41.0%  46.1%   0%  0%     0%  0%  0%  
1  Curiel, Sarah Anastasia  2.758    40%    15%    25%  10%  5%     5%  0%  0%  
2     Given, William Allan  2.580  6.25%  43.7%  43.7%   0%  0%  6.25%  0%  0%  
3        McCarthy, Jenna L  2.426  12.5%  27.5%    45%   5%  5%     5%  0%  0%  
4        Miller, Ronald L.  2.718  25.6%  28.2%  46.1%   0%  0%     0%  0%  0%  


In [33]:

course_repeated_counts = df.groupby(['Instructor','Subject', 'Course']).size().reset_index(name='Times_Taught')

repeated_prof = course_repeated_counts[course_repeated_counts['Times_Taught'] > 5]
repeated_prof = repeated_prof.sort_values(by = 'Times_Taught', ascending=False)

print(repeated_prof)

course_name = df.drop_duplicates(subset = ['Subject', 'Course'])[['Subject', 'Course', 'Title']]
final= pd.merge(repeated_prof,course_name, on = ['Subject', 'Course'], how='left')
final=final[['Instructor','Subject', 'Course', 'Title', 'Times_Taught']]

print(final.to_string(index=False))

final.to_csv('repeat_instructors.csv', index=False)
print("\nSaved to 'repeat_instructors.csv'")

                       Instructor Subject Course  Times_Taught
1449      Campbell, Robert Harold     MGT     16            26
6660           Moran, Cathleen L.     MGT    166            26
8803             Seshadri, Kartik     MUS    95W            26
4096             Hedges, Kathleen     MGT    172            26
392       Armour, Jon Christopher    BIPN    105            26
...                           ...     ...    ...           ...
5903                  Lu, Weijing    HIEA    138             6
1894        Cleland, Elsa Eleanor    BIEB    174             6
8065   Ribas Norris, Janet Vanesa    SOCI    111             6
5921               Lubarda, Marko     MAE    30B             6
11152            duBois, Page Ann     MMW     11             6

[1262 rows x 4 columns]
                         Instructor Subject Course                          Title  Times_Taught
            Campbell, Robert Harold     MGT     16        Personal Ethics at Work            26
                 Moran, Cat

In [34]:
unique_prof = final['Instructor'].unique()
print(unique_prof)

['Campbell, Robert Harold' 'Moran, Cathleen L.' 'Seshadri, Kartik'
 'Hedges, Kathleen' 'Armour, Jon Christopher' 'Anderson, Kenneth'
 'Wastal, Carrie King' 'Guirguis, Mark C' 'McKay, Michael J'
 'Harting, Carla Kelso' 'McKay, Mary A' 'Bates, Kenneth Walter'
 'Pecore, Joseph P.' 'Johnson, Christine M.' 'Ehrich, Kristine R'
 'Walsh, Kim C.' 'McCleary, Keith Long' 'Young, Mark T' 'Dunn, Louis Jean'
 'Goodall, Grant Todd' 'Kenyatta, Kamau' 'Lowe, Angela Beth'
 'Houskeeper, Robert' 'Wilson, Terry A.' 'Salovey, Todd'
 'Buchanan, Harold G' 'Somasundaram, Tyagarajan' 'Day, Christopher D'
 'Kokinis, Troy Andreas' 'Eldon, John' 'Eberhard, Craig A'
 'Anderson, John Charles' 'Warstadt, Melissa Fawn' 'Caligagan, Tony'
 'Zimmermann, Carsten' 'Perols, Johan Lars' 'Wagner, Timothy Ryan'
 'Meyer, Alison Bloomfield' 'Pirino, Giorgia' 'Baghdadchi, Saharnaz'
 'Ngo, Anh-Thu T.' 'Munoz Sanchez, Alicia' 'Lott, Margaret S.'
 'Twohig, Niall Ivan' 'Gopal, Christopher S.' 'Tustaniwskyj, Jerry I'
 'Jun, Grace Shi

In [37]:

import pandas as pd
import fitz
import re
from collections import defaultdict

pdf = fitz.open("professor_directory_emails.pdf")
records = []

for page in pdf:
    rows= defaultdict(lambda: {"name": [], "email": []})
    for i in page.get_text("words"):
        x0, y0, x1, y1, text, *_ =i
        if y0 < 70:
            continue
        row=rows[round(y0, 1)]
        if x0<165:
            row["name"].append((x0, text))
        elif x0 >= 668:
            row["email"].append((x0, text))

    for row in rows.values():
        name =" ".join(text for x, text in sorted(row["name"]))
        email = "".join(text for x, text in sorted(row["email"]))
        if re.fullmatch(r"[\w.+-]+@ucsd\.edu", email, re.I):
            records.append({"Instructor": name,"Email": email})
pdf.close()

directory = pd.DataFrame(records)
#last and first name
def prof_name(name):
    if pd.isna(name):
        return ""
    name = str(name).strip().lower()
    if "," not in name:
        return ""
    last, first=name.split(",", 1)
    first =first.strip().split()
    if not first:
        return ""
    clean = lambda s: re.sub(r"[^a-z]", "", s)
    return clean(last) + "|" + clean(first[0])

directory["match_key"] = directory["Instructor"].apply(prof_name)
final["match_key"] = final["Instructor"].apply(prof_name)

#unique email
unique_email_counts= directory.groupby("match_key")["Email"].nunique()
unique_keys_email = unique_email_counts[unique_email_counts == 1].index

email_lookup=(directory[directory["match_key"].isin(unique_key_email)]
    .drop_duplicates(subset="match_key")
    [["match_key", "Email"]])

# emails and professors
final = final.drop(columns= ["Email"], errors="ignore")
final = final.merge(email_lookup, on = "match_key", how="left")
final = final.drop(columns=["match_key"])

print(final.to_string(index=False))

print("Total rows:", len(final))
print("Rows with matched emails:", final["Email"].notna().sum())
print("Rows without matched emails:", final["Email"].isna().sum())

final.to_csv("repeat_instructors_with_emails.csv", index=False)
print("Saved to repeat_instructors_with_emails.csv")

                         Instructor Subject Course                          Title  Times_Taught                     Email
            Campbell, Robert Harold     MGT     16        Personal Ethics at Work            26       rhcampbell@ucsd.edu
                 Moran, Cathleen L.     MGT    166     Corp Social Responsibility            26           cmoran@ucsd.edu
                   Seshadri, Kartik     MUS    95W          World Music Ensembles            26       kaseshadri@ucsd.edu
                   Hedges, Kathleen     MGT    172    Business Project Management            26                       NaN
            Armour, Jon Christopher    BIPN    105          Animal Physiology Lab            26                       NaN
                  Anderson, Kenneth     MUS    95G                   Gospel Choir            26       k7anderson@ucsd.edu
                Wastal, Carrie King    MCWP     40               Critical Writing            26          cwastal@ucsd.edu
                   Guirg